# Policy-priority modeling: predict intervention segment

This notebook predicts the synthetic `policy_priority_segment`. The purpose is to show how a social-service team could route households into different intervention pathways, such as prevention support, housing emergency response, or benefit navigation.

The model excludes final labels and direct outcome fields so the example behaves like an early-routing model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("synthetic_food_housing_insecurity_100k.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_food_housing_insecurity_100k.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Prepare features and target

The target has five synthetic policy-priority segments. The features are demographic, economic, access, housing, and health indicators.

In [ ]:
TARGET = "policy_priority_segment"
LEAKAGE_COLUMNS = [
    "household_id",
    "train_test_split",
    TARGET,
    "overall_hardship_score",
    "food_security_status",
    "food_insecure_label",
    "housing_insecure_label",
    "meals_skipped_last_30d",
    "months_food_shortage_last_year",
    "pantry_use_last_30d",
]

features = [c for c in df.columns if c not in LEAKAGE_COLUMNS]
train_mask = df["train_test_split"].eq("train")

X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, TARGET]
X_test = df.loc[~train_mask, features]
y_test = df.loc[~train_mask, TARGET]

print(y_train.value_counts(normalize=True).round(3))

## 2. Fit a multi-class model

Logistic regression is used for a transparent, fast baseline. In production, this should be compared with tree-based and calibrated models.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in features if c not in categorical_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]), categorical_cols),
])

segment_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, solver="saga", class_weight="balanced", random_state=42)),
])
segment_model.fit(X_train, y_train)
print("Model trained.")

## 3. Evaluate segment prediction

Accuracy is useful, but the confusion matrix is more important because policy errors have different consequences by segment.

In [ ]:
pred_segment = segment_model.predict(X_test)
pred_proba = segment_model.predict_proba(X_test)
classes = segment_model.named_steps["model"].classes_

print("Accuracy:", round(accuracy_score(y_test, pred_segment), 3))
print(classification_report(y_test, pred_segment, digits=3))

cm = pd.DataFrame(confusion_matrix(y_test, pred_segment, labels=classes), index=classes, columns=classes)
cm

## 4. Turn predictions into policy actions

This simple mapping demonstrates how model output can become an operational queue. The mapping is illustrative and should be reviewed by domain experts.

In [ ]:
ACTION_MAP = {
    "stable_or_low_need": "Monitor; provide general resource information.",
    "prevention_support": "Offer light-touch prevention support and budgeting resources.",
    "benefit_navigation": "Prioritize SNAP/WIC/school-meal screening and application help.",
    "housing_emergency": "Route to rental assistance, eviction prevention, and utility support.",
    "severe_multiple_hardship": "Assign intensive case management and multi-benefit coordination.",
}

priority_rank = {
    "severe_multiple_hardship": 1,
    "housing_emergency": 2,
    "benefit_navigation": 3,
    "prevention_support": 4,
    "stable_or_low_need": 5,
}

proba_df = pd.DataFrame(pred_proba, columns=[f"prob_{c}" for c in classes], index=X_test.index)
queue = pd.concat([
    df.loc[~train_mask, ["household_id", "region", "state", "urbanicity"]].reset_index(drop=True),
    pd.DataFrame({
        "actual_segment": y_test.values,
        "predicted_segment": pred_segment,
        "confidence": pred_proba.max(axis=1),
    }),
    proba_df.reset_index(drop=True),
], axis=1)
queue["recommended_action"] = queue["predicted_segment"].map(ACTION_MAP)
queue["priority_rank"] = queue["predicted_segment"].map(priority_rank)
queue = queue.sort_values(["priority_rank", "confidence"], ascending=[True, False])
queue.head(20)

## 5. Inspect model drivers by class

The table lists the largest absolute coefficients across all classes. Coefficients are descriptive model signals, not causal estimates.

In [ ]:
feature_names = segment_model.named_steps["preprocess"].get_feature_names_out()
coef = segment_model.named_steps["model"].coef_

rows = []
for class_name, class_coef in zip(classes, coef):
    top_idx = np.argsort(np.abs(class_coef))[-10:][::-1]
    for idx in top_idx:
        rows.append({
            "class": class_name,
            "feature": feature_names[idx],
            "coefficient": class_coef[idx],
            "abs_coefficient": abs(class_coef[idx]),
        })

coef_by_class = pd.DataFrame(rows).sort_values(["class", "abs_coefficient"], ascending=[True, False])
coef_by_class.head(30)

## 6. Save priority queue sample

This file can be reviewed in spreadsheet form or connected to a dashboard.

In [ ]:
queue.head(10000).to_csv("policy_priority_queue_sample.csv", index=False)
Path("policy_priority_queue_sample.csv")